# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic dataset info
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll display all record sets, their `@id`, and their fields (`cr:field` with `@id`) for exploration.

In [ ]:
# Examine record sets and field ids

def pprint_record_sets(meta):
    if hasattr(meta, 'record_sets'):
        for rs in meta.record_sets:
            print(f"RecordSet: {rs['@id']}  (name: {rs.get('name', '<unnamed>')})")
            if 'field' in rs:
                print("  Fields:")
                for field in rs['field']:
                    if isinstance(field, dict):
                        print(f"   - {field.get('@id', str(field))} (name: {field.get('name', '<unnamed>')})")
                    else:
                        print(f"   - {field}")
            print()
    else:
        # Fallback: try to access _metadata dict
        meta_dict = meta.to_json() if hasattr(meta, 'to_json') else meta
        record_sets = meta_dict.get('recordSet', [])
        for rs in record_sets:
            rs_id = rs.get('@id', '<unknown>')
            print(f"RecordSet: {rs_id} (name: {rs.get('name', '<unnamed>')})")
            fields = rs.get('field', [])
            if fields:
                print("  Fields:")
                for field in fields:
                    if isinstance(field, dict):
                        print(f"   - {field.get('@id', str(field))} (name: {field.get('name', '<unnamed>')})")
                    else:
                        print(f"   - {field}")
            print()

pprint_record_sets(metadata)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from above.

> **Note:** If you know the primary survey record set `@id`, set it below, e.g., `'cr:survey_responses'`. You may need to inspect the overview above to get the right IDs.

In [ ]:
# Identify the main record sets for analysis
# If the schema does not provide embedded recordSet details, you may need to use known @ids from schema or schema fields
# Example primary record set ID for our FAIR2 survey data (replace if necessary):

main_record_set_id = 'cr:survey_responses'  # Adjust this to the correct @id from record set overview if different

# For demonstration, collect all available record set IDs
meta_dict = metadata.to_json() if hasattr(metadata, 'to_json') else metadata
record_sets_meta = meta_dict.get('recordSet', [])
record_set_ids = [rs['@id'] for rs in record_sets_meta]
print('Available record set @ids:', record_set_ids)

# Attempt to load data for each record set, reporting if any fail
dataframes = {}
for rid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rid))
        df = pd.DataFrame(records)
        print(f"{rid} -- loaded with {len(df)} rows and columns: {list(df.columns)}\n")
        dataframes[rid] = df
    except Exception as e:
        print(f"Could not load records for {rid}:", str(e))

# Choose the primary survey record set for further analysis
primary_rid = main_record_set_id if main_record_set_id in dataframes else record_set_ids[0] if record_set_ids else None
if primary_rid:
    print(f"Using record set: {primary_rid}")
    print(dataframes[primary_rid].head())
else:
    print('No valid record set found.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by specific criteria, normalizing numeric fields, and grouping by key attributes for further analysis.

We'll pick a key numeric field (for instance PHQ-9 or GAD-7 score field) from the primary record set. Set the appropriate field's `@id` below (from section 2 or 3).

In [ ]:
# Specify the numeric field by its @id used in the data (e.g. 'phq9_score', 'GAD7_score', or full @id reference)
# Replace with actual column name printed in previous output if necessary

numeric_field = 'phq9_score'  # E.g., PHQ-9 questionnaire total score
group_field = 'level_of_education'  # Grouping variable (e.g., 'gender' or 'level_of_education')
threshold = 10

primary_df = dataframes[primary_rid] if primary_rid in dataframes else None

if primary_df is not None and numeric_field in primary_df.columns:
    # Filter numeric_field > threshold
    filtered_df = primary_df[primary_df[numeric_field].astype(float) > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (count={len(filtered_df)}):")
    display_cols = [col for col in [numeric_field, group_field] if col in filtered_df.columns]
    print(filtered_df[display_cols].head())

    # Normalize the numeric field
    filtered_df[numeric_field + '_normalized'] = (
        (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) /
        filtered_df[numeric_field].astype(float).std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Group by group_field
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field} not found in dataframe or no data available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For example, plot the distribution of PHQ-9 scores and compare by education level (or other grouping).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_df is not None and numeric_field in primary_df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(primary_df[numeric_field].astype(float), kde=True, bins=20)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group (if available)
    if group_field in primary_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=primary_df[group_field], y=primary_df[numeric_field].astype(float))
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<ul>
<li>The dataset contains survey-based mental health indicators with demographic data from Kilifi County, Kenya.</li>
<li>You can access record sets, fields, and filter/analyze data by referencing the schema's `@id`s for record sets and fields.</li>
<li>Use the `mlcroissant` library to seamlessly parse schema and iterate over records for analysis as done above.</li>
<li>Further analysis could include model building, advanced statistics, or integration with other regional datasets.</li>
</ul>